In [1]:
# =========================================
# CUSTOMER CHURN PREDICTION - FINAL PIPELINE
# =========================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# -----------------------------
# 2. LOAD DATASETS
# -----------------------------

train_df = pd.read_csv("customer_churn_dataset-training-master.csv")
test_df = pd.read_csv("customer_churn_dataset-testing-master.csv")

# -----------------------------
# 3. REMOVE USELESS COLUMN
# -----------------------------

train_df = train_df.drop('CustomerID', axis=1)
test_df = test_df.drop('CustomerID', axis=1)

# -----------------------------
# 4. HANDLE MISSING VALUES
# -----------------------------

# Fill categorical columns using mode

for col in train_df.select_dtypes(include='object').columns:
    train_df[col] = train_df[col].fillna(
        train_df[col].mode()[0]
    )

for col in test_df.select_dtypes(include='object').columns:
    test_df[col] = test_df[col].fillna(
        test_df[col].mode()[0]
    )

# Fill numerical columns using median

for col in train_df.select_dtypes(include=['int64', 'float64']).columns:
    train_df[col] = train_df[col].fillna(
        train_df[col].median()
    )

for col in test_df.select_dtypes(include=['int64', 'float64']).columns:
    test_df[col] = test_df[col].fillna(
        test_df[col].median()
    )

# -----------------------------
# 5. CHECK NULL VALUES
# -----------------------------

print("Remaining Null Values in Train:")
print(train_df.isnull().sum())

print("\nRemaining Null Values in Test:")
print(test_df.isnull().sum())

# -----------------------------
# 6. DEFINE FEATURES & TARGET
# -----------------------------

X_train = train_df.drop('Churn', axis=1)
y_train = train_df['Churn']

X_test = test_df.drop('Churn', axis=1)
y_test = test_df['Churn']

# -----------------------------
# 7. CATEGORY ORDER
# -----------------------------

subscription_order = ['Basic', 'Standard', 'Premium']

contract_order = ['Monthly', 'Quarterly', 'Annual']

# -----------------------------
# 8. NUMERICAL FEATURES
# -----------------------------

numerical_features = [
    'Age',
    'Tenure',
    'Usage Frequency',
    'Support Calls',
    'Payment Delay',
    'Total Spend',
    'Last Interaction'
]

# -----------------------------
# 9. PREPROCESSOR
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[

        # Scale Numerical Features
        (
            'num',
            StandardScaler(),
            numerical_features
        ),

        # Ordinal Encoding
        (
            'ordinal_sub',
            OrdinalEncoder(categories=[subscription_order]),
            ['Subscription Type']
        ),

        (
            'ordinal_con',
            OrdinalEncoder(categories=[contract_order]),
            ['Contract Length']
        ),

        # One Hot Encoding
        (
            'onehot_gender',
            OneHotEncoder(
                sparse_output=False,
                handle_unknown='ignore'
            ),
            ['Gender']
        )

    ],

    remainder='passthrough'
)

# -----------------------------
# 10. APPLY PREPROCESSING
# -----------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

# -----------------------------
# 11. CHECK FOR NaN VALUES
# -----------------------------

print("\nNaN values in X_train_processed:")
print(np.isnan(X_train_processed).sum())

print("\nNaN values in X_test_processed:")
print(np.isnan(X_test_processed).sum())

# -----------------------------
# 12. TRAIN MODEL
# -----------------------------

model = LogisticRegression(max_iter=1000)

model.fit(X_train_processed, y_train)

# -----------------------------
# 13. MAKE PREDICTIONS
# -----------------------------

y_pred = model.predict(X_test_processed)

# -----------------------------
# 14. ACCURACY SCORE
# -----------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy Score:")
print(accuracy)

# -----------------------------
# 15. CONFUSION MATRIX
# -----------------------------

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

# -----------------------------
# 16. CLASSIFICATION REPORT
# -----------------------------

report = classification_report(y_test, y_pred)

print("\nClassification Report:")
print(report)

# -----------------------------
# 17. FEATURE COEFFICIENTS
# -----------------------------

feature_names = preprocessor.get_feature_names_out()

coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
})

coefficients = coefficients.sort_values(
    by='Coefficient',
    ascending=False
)

print("\nFeature Coefficients:")
print(coefficients)

# -----------------------------
# 18. TRAIN VS TEST SCORE
# -----------------------------

train_score = model.score(X_train_processed, y_train)

test_score = model.score(X_test_processed, y_test)

print("\nTrain Score:", train_score)

print("Test Score:", test_score)

Remaining Null Values in Train:
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

Remaining Null Values in Test:
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

NaN values in X_train_processed:
0

NaN values in X_test_processed:
0

Accuracy Score:
0.5792089974213191

Confusion Matrix:
[[ 7169 26712]
 [  376 30117]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.21      0.35     33881
           1       0.53      0.99      0.69     30493

    accuracy                           0.58     64374
   ma